In [ ]:

# Lower bound determined by noise...
from util.DataGen import *
from util.Processing import *
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['figure.figsize'] = [15, 7]

snr = np.logspace(-1, 1, 100)  # SNR = sig/noise
sig_base_strength = 1
base_noises = sig_base_strength / snr

fig, axes = plt.subplots(5,1, figsize=(10, 8), dpi=400)
axes[0].loglog(snr, base_noises)
axes[0].set_xlabel('SNR')
axes[0].set_ylabel('Relative Noise [dB]')
axes[0].set_title('Noise Levels')

times, pulse = nai_pulse(1)
axes[1].plot(pulse)

N = 1000
n = len(pulse)
n_pulses = 5

np.random.seed(0)
shifts = np.random.randint(0, N-n, n_pulses)
shifts = np.sort(shifts)
scale = np.random.randint(1, 10, n_pulses)

for shift, energy in zip(shifts, scale):
    axes[2].plot([shift, shift], [0, energy])
energies = np.zeros(N)
energies[shifts] = scale
axes[2].set_xlim([0, len(energies)])

conv = fft_convolve(energies, pulse, 0)[0:len(energies)]
axes[3].plot(conv)
axes[3].set_xlim([0, len(energies)])

deconv = fft_deconvolve(conv, pulse, 0)[0:len(energies)]
axes[4].plot(deconv)
axes[4].set_xlim([0, len(energies)])

In [ ]:
snr = np.logspace(1, 3, 3)  # SNR = sig/noise
sig_base_strength = 1
base_noises = sig_base_strength / snr

fig, axes = plt.subplots(2, 1, figsize=(10, 4), dpi=400)
fig.tight_layout()
axes[0].loglog(snr, base_noises, marker='*')
axes[0].set_xlabel('SNR')
# axes[0].set_ylabel('Relative Noise [dB]')
axes[0].set_title('Noise Levels')

times, pulse = nai_pulse(1)
axes[1].plot(pulse, label='kernel')

N = 200
n = len(pulse)
n_pulses = 1

np.random.seed(0)
shifts = np.array([10, 20, 45, 50, 80, 81, 82, 125])
scale = np.array([1, 1, 1, 1, 1, 1, 2 ,2])

energies = np.zeros(N)
energies[shifts] = scale

axes[1].set_xlim([0, len(energies)])
axes[1].set_title('Energy and Pulse to convolve')

conv = fft_convolve(energies, pulse, 10000)[0:len(energies)]
axes[1].plot(conv, label='Convolved')
axes[1].set_xlim([0, len(energies)])
axes[1].set_title('Convolved')
axes[1].legend(loc=7)

for shift, energy in zip(shifts, scale):
    axes[1].plot([shift, shift], [0, energy], 'r')

fig2, axes2 = plt.subplots(3, 1, figsize=(10, 8), dpi=400)

for i, noise_stdev in enumerate(base_noises):
    # print(noise_stdev)
    
    label='noise stdev={}'.format(noise_stdev)
    
    # conv_noise = conv + np.random.normal(0, noise_stdev, N)
    conv_noise = conv + np.random.randn(N) * noise_stdev
    deconv = fft_deconvolve(conv_noise, pulse, 0, signal_fft_ax=axes2[0], signal_label=label,
                            kernel_fft_ax=axes2[0], kernel_label='kernel fft')[0:len(energies)]
    
    axes2[1].plot(deconv, label=label, alpha=.5)
    deconv /= np.max(deconv)
    axes2[2].plot(deconv, label=label, alpha=.5)
    
axes2[1].set_xlim([0, len(energies)])
axes2[1].set_title('Deconvolved Signal')
axes2[1].legend(loc=7)

axes2[0].set_title('Kernel and Signal FFT Spectra')
axes2[0].legend()
# axes2[0].legend(loc=7)

axes2[2].set_xlim([0, len(energies)])
axes2[2].set_title('Normalized Deconvolved Signal')
axes2[2].legend(loc=7)

In [ ]:
# Apply filters
snr = np.logspace(1, 3, 3)  # SNR = sig/noise
sig_base_strength = 1
base_noises = sig_base_strength / snr

fig, axes = plt.subplots(1 + len(snr), 1, figsize=(10, 8), dpi=400)
fig.tight_layout()
times, pulse = nai_pulse(1)
print(len(pulse))
axes[0].plot(pulse, label='Kernel')

N = 200
pulse = np.concatenate((pulse, np.zeros(N-len(pulse))), axis=0)
n = len(pulse)
n_pulses = 1

np.random.seed(0)
shifts = np.array([10, 20, 45, 50, 80, 81, 82, 125])
scale = np.array([1, 1, 1, 1, 1, 1, 2, 2])

energies = np.zeros(N)
energies[shifts] = scale

axes[0].set_xlim([0, len(energies)])
axes[0].set_title('Energy and Pulse to convolve')

conv = fft_convolve(energies, pulse, 10000)[0:len(energies)]
axes[0].plot(conv, label='Convolved')
axes[0].set_xlim([0, len(energies)])
axes[0].set_title('Convolved')
axes[0].legend(loc=7)

for shift, energy in zip(shifts, scale):
    axes[0].plot([shift, shift], [0, energy], 'r')

fc = 60
butterworth = signal.butter(3, fc, btype='lowpass', analog=False, output='sos', fs=N)
chebyshev1 = signal.cheby1(3, 20, fc, btype='lowpass', analog=False, output='sos', fs=N) # param 2: The maximum ripple allowed below unity gain in the passband. Specified in decibels, as a positive number.
chebyshev2 = signal.cheby2(3, 20, fc, btype='lowpass', analog=False, output='sos', fs=N)  # param2: The minimum attenuation required in the stop band. Specified in decibels, as a positive number.

# Combining the above...
elliptic = signal.ellip(3, 3, 100, fc, btype='lowpass', analog=False, output='sos', fs=N) # param 2: The maximum ripple allowed below unity gain in the passband. Specified in decibels, as a positive number.
                                                                               # param 3: The maximum ripple allowed below unity gain in the passband. Specified in decibels, as a positive number
# wiener_filter = signal.wiener()
labels=['Butterworth',
        # 'Chebyshev1',
        'Chebyshev2',
        'Elliptic']
filters = [butterworth,
           # chebyshev1,
           chebyshev2,
           elliptic]

for i, noise_stdev in enumerate(base_noises):
    # print(noise_stdev)

    title='noise stdev={}'.format(noise_stdev)
    axes[i+1].set_title(title)

    conv_noise = conv + np.random.normal(0, noise_stdev, N)

    deconv = fft_deconvolve(conv_noise, pulse, 0)[0:len(energies)]
    deconv /= np.max(deconv)
    
    alpha = .5
    axes[i+1].plot(deconv, label='Unfiltered', alpha=alpha)

    for j, filter_ in enumerate(filters):
        filtered_signal = signal.sosfilt(filter_, conv_noise)
        filtered_kernel = signal.sosfilt(filter_, pulse)

        deconv = fft_deconvolve(filtered_signal, pulse, 0)[0:len(energies)]
        deconv /= np.max(deconv)
        axes[i+1].plot(deconv, label=labels[j], alpha=alpha)
        axes[i+1].set_xlim([0, len(deconv)])

for ax in axes:
    ax.legend(loc=7)


In [ ]:
n_snr = 4
snr = np.logspace(1, n_snr, n_snr)  # SNR = sig/noise
sig_base_strength = 1
base_noises = sig_base_strength / snr

fig, axes = plt.subplots(2 + len(snr), 1, figsize=(10, 8), dpi=400)
fig.tight_layout()
axes[0].loglog(snr, base_noises, marker='*')
axes[0].set_xlabel('SNR')
# axes[0].set_ylabel('Relative Noise [dB]')
axes[0].set_title('Noise Levels')

times, pulse = nai_pulse(1)
axes[1].plot(pulse, label='kernel')

N = 200
n = len(pulse)
n_pulses = 1

np.random.seed(0)
shifts = np.array([10, 20, 45, 50, 80, 81, 82, 125])
scale = np.array([1, 1, 1, 1, 1, 1, 2 ,2])

energies = np.zeros(N)
energies[shifts] = scale

axes[1].set_xlim([0, len(energies)])
axes[1].set_title('Energy and Pulse to convolve')

conv = fft_convolve(energies, pulse, 10000)[0:len(energies)]
axes[1].plot(conv, label='Convolved')
axes[1].set_xlim([0, len(energies)])
axes[1].set_title('Convolved')
axes[1].legend(loc=7)

for shift, energy in zip(shifts, scale):
    axes[1].plot([shift, shift], [0, energy], 'r')
    
# Y = AX + b :: convolved = mixing * deconvolved + (noise + residual from previous block of pulses)
c = np.concatenate((pulse, np.zeros(N - len(pulse))), axis=0)
from scipy.linalg import circulant
A = circulant(c)
A_inv = np.linalg.inv(A)

for i, noise_stdev in enumerate(base_noises):
    print(noise_stdev)
    conv_noise = conv + np.random.normal(0, noise_stdev, N)

    X = A_inv @ conv_noise 
    X  /= np.max(X)
    
    alpha = .5
    right = axes[2+i].twinx()
    right.plot(conv_noise, alpha=.5*alpha, color='orange', label = 'Noisy Timeseries, stdev = {}'.format(noise_stdev))

    axes[2+i].plot(X, label='Deconv from voltage series', alpha=alpha)
    axes[2+i].legend(loc=7)
    axes[2+i].set_xlim([0, N])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn import datasets, linear_model

# fig, axes = plt.subplots(len(snr), 1, figsize=(10, 8), dpi=400)
# 
# # Y = AX +b :: convolved = mixing * deconvolved + (noise + residual from previous block of pulses)
# c = np.concatenate((pulse, np.zeros(N - len(pulse))), axis=0)
# from scipy.linalg import circulant
# A = circulant(c)
# A_inv = np.linalg.inv(A)
# 
# for i, noise_stdev in enumerate(base_noises):
#     print(noise_stdev)
#     conv_noise = conv + np.random.normal(0, noise_stdev, N)
# 
#     X = A_inv @ conv_noise 
#     X  /= np.max(X)
#     
#     alpha = .5
#     right = axes[2+i].twinx()
#     right.plot(conv_noise, alpha=.5*alpha, color='orange')
# 
#     axes[2+i].plot(X, label='noise stdev = {}'.format(noise_stdev), alpha=alpha)
#     axes[2+i].legend(loc=7)
#     axes[2+i].set_xlim([0, N])
#     

In [ ]:
import scipy.linalg

# Lower bound determined by noise...
from functions import *
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['figure.figsize'] = [15, 7]

fig, axes = plt.subplots(3,1, figsize=(10, 6), dpi=400)
fig.tight_layout()

times, pulse = nai_pulse(1)
# axes[0].plot(pulse)

N = 1000
n = len(pulse)
n_pulses = 500

np.random.seed(0)
shifts = np.random.randint(0, N-n, n_pulses)
shifts = np.sort(shifts)
scale = np.random.randint(1, 10, n_pulses)

for shift, energy in zip(shifts, scale):
    axes[0].plot([shift, shift], [0, energy])
energies = np.zeros(N)
energies[shifts] += scale
axes[0].set_xlim([0, len(energies)])
axes[0].set_title('High Count Rate')

conv = fft_convolve(energies, pulse, 0)[0:len(energies)]
axes[1].plot(conv)
axes[1].set_xlim([0, len(energies)])

deconv = fft_deconvolve(conv, pulse, 0)[0:len(energies)]
for shift, energy in zip(np.arange(0, deconv.size), deconv):
    axes[2].plot([shift, shift], [0, energy])
axes[2].set_xlim([0, len(energies)])

error = np.sum(energies - deconv) / len(energies) + .000001
axes[2].set_title('Deconvolution energies: {e:.2f}% Error'.format(e=error))
# Note that this error is relative to not to the original energies, but relative to their sum
# this means that pulses of coincident index have already been summed together


# Two photons of adjacent index
fig, axes = plt.subplots(3,1, figsize=(10, 6), dpi=400)
fig.tight_layout()

N = 1000
shifts = [100, 101, 300, 301, 500, 501]
scale = [5, 5, 1, 10, 10, 1]

for shift, energy in zip(shifts, scale):
    axes[0].plot([shift, shift], [0, energy])
energies = np.zeros(N)
energies[shifts] += scale
axes[0].set_xlim([0, len(energies)])
axes[0].set_title('Adjacent Pulses')

conv = fft_convolve(energies, pulse, 0)[0:len(energies)]
axes[1].plot(conv)
axes[1].set_xlim([0, len(energies)])

deconv = fft_deconvolve(conv, pulse, 0)[0:len(energies)]
for shift, energy in zip(np.arange(0, deconv.size), deconv):
    axes[2].plot([shift, shift], [0, energy])
axes[2].set_xlim([0, len(energies)])

error = np.abs(np.sum(energies - deconv)) / len(energies) + .00001
axes[2].set_title('Deconvolution energies: {e:.2f}% Error'.format(e=error))


In [ ]:
import scipy.linalg

# Lower bound determined by noise...
from functions import *
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['figure.figsize'] = [15, 7]

fig, axes = plt.subplots(3,1, figsize=(10, 8), dpi=400)
fig.tight_layout()

times, pulse = nai_pulse(1)

N = 1000
n = len(pulse)
n_pulses = 100

np.random.seed(0)
# shifts = np.random.randint(0, N-n, n_pulses)
shifts = np.random.randint(0, N, n_pulses)
shifts = np.sort(shifts)
scale = np.random.uniform(1, 4, n_pulses)
scale = 10 ** scale
energies = np.zeros(N)
for i in range(n_pulses):
    energies[shifts[i]] += scale[i]
energies += 1
    
axes[0].plot(energies)
axes[0].set_xlim([0, len(energies)])
axes[0].set_title('Energies')
axes[0].set_yscale('log')

conv = fft_convolve(energies, pulse, 0)[0:len(energies)]
axes[1].plot(conv)
axes[1].set_xlim([0, len(energies)])
axes[1].set_title('Trace')
axes[1].set_yscale('log')

deconv = fft_deconvolve(conv, pulse, 0)[0:len(energies)]
print(len(energies), len(deconv))
error = np.sum(energies-deconv)
print('error', error)
axes[2].plot(deconv)
axes[2].set_xlim([0, len(energies)])
axes[2].set_title('Reconstruction Error: {:.5f}'.format(error))
axes[2].set_yscale('log')